# 02 Exploratory Analysis

Component 2 Phase 1 research notebook.

## 1. EDA Purpose
This notebook examines multilingual and distributional properties of the pilot comments. Script detection is used only to identify writing systems, not language. Latin-script comments may be English, Singlish, Tanglish, names, URLs, or other content.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not ((PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT / 'shared').exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'research' / 'component_2' / 'src'
sys.path.insert(0, str(SRC_DIR))

COMMENTS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'component_2' / 'component2_pilot_comments.csv'
VIDEOS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'component_2' / 'component2_pilot_videos.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'component_2'
ANALYSIS_DIR = PROJECT_ROOT / 'data' / 'analysis' / 'component_2' / '01_pilot_2000_comments'
CHART_DIR = ANALYSIS_DIR / 'charts'
for directory in [PROCESSED_DIR, ANALYSIS_DIR, CHART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

from data_io import load_comments, load_videos, parse_datetime_columns
from validation import comments_per_channel, comments_per_video, reply_summary, video_channel_distribution
from eda import add_text_features, comments_per_video_summary, potential_spam_summary, publication_timeline, save_bar_chart, save_histogram

comments = parse_datetime_columns(load_comments(COMMENTS_PATH), ['published_at', 'edited_at'])
videos = parse_datetime_columns(load_videos(VIDEOS_PATH), ['published_at'])
comments_features = add_text_features(comments)

## 2. Comments and Videos per Channel

In [2]:
channel_comments = comments_per_channel(comments)
channel_videos = video_channel_distribution(videos)
display(channel_comments)
display(channel_videos)
save_bar_chart(channel_comments.set_index('channel_name')['total_comments'], 'EDA Comments per Channel', 'Comments', CHART_DIR / 'eda_comments_per_channel.png')
save_bar_chart(channel_videos.set_index('channel_name')['total_videos'], 'EDA Videos per Channel', 'Videos', CHART_DIR / 'eda_videos_per_channel.png')

,channel_id,channel_name,total_comments,videos_with_comments
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,400,76
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,400,38
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,400,43
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,400,50
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,400,48


,channel_id,channel_name,total_videos
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,100
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,76
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,51
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,50
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,50


PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/eda_videos_per_channel.png')

## 3. Comments per Video

In [3]:
per_video = comments_per_video(comments)
display(per_video.head(10))
display(comments_per_video_summary(comments))
save_histogram(per_video['total_comments'], 'Comments per Video', 'Comments', CHART_DIR / 'eda_comments_per_video_histogram.png', bins=25)

,channel_id,video_id,total_comments
148,UCOPPH1_whpThgY6e7r3qg-w,jyqNxhhnRGw,16
152,UCOPPH1_whpThgY6e7r3qg-w,tV-r0qTmKKo,16
115,UCOPPH1_whpThgY6e7r3qg-w,15oxxeqi04o,16
118,UCOPPH1_whpThgY6e7r3qg-w,2h_4xF2JRBY,16
120,UCOPPH1_whpThgY6e7r3qg-w,4dihtmEODXY,16
156,UCOPPH1_whpThgY6e7r3qg-w,y3oE_AQj7ws,16
124,UCOPPH1_whpThgY6e7r3qg-w,9TMve7LWrsQ,16
139,UCOPPH1_whpThgY6e7r3qg-w,X9v0iKUI23k,16
133,UCOPPH1_whpThgY6e7r3qg-w,LtK0NKlGwZk,16
129,UCOPPH1_whpThgY6e7r3qg-w,HyxxpnFNYd8,15


count    255.000000
mean       7.843137
std        4.743718
min        1.000000
25%        3.000000
50%        7.000000
75%       12.000000
max       16.000000
Name: comment_id, dtype: float64

PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/eda_comments_per_video_histogram.png')

## 4. Comment Length and Word Count Distributions

In [4]:
display(comments_features[['char_length', 'word_count']].describe())
save_histogram(comments_features['char_length'], 'Comment Character Length', 'Characters', CHART_DIR / 'eda_character_length_histogram.png')
save_histogram(comments_features['word_count'], 'Comment Word Count', 'Words', CHART_DIR / 'eda_word_count_histogram.png')

,char_length,word_count
count,2000.000000,2000.000000
mean,52.857500,8.446000
std,141.647521,21.812725
min,1.000000,1.000000
25%,9.000000,1.000000
50%,26.000000,4.000000
75%,55.000000,9.000000
max,3238.000000,486.000000


PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/eda_word_count_histogram.png')

## 5. Publication Timeline

In [5]:
comment_timeline = publication_timeline(comments, 'published_at')
video_timeline = publication_timeline(videos, 'published_at')
display(comment_timeline.tail(12))
display(video_timeline.tail(12))

/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/research/component_2/src/eda.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return dates.dt.to_period(freq).value_counts().sort_index().rename_axis("period").reset_index(name="count")
/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/research/component_2/src/eda.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return dates.dt.to_period(freq).value_counts().sort_index().rename_axis("period").reset_index(name="count")


,period,count
0,2026-01,148
1,2026-02,347
2,2026-03,12
3,2026-04,27
4,2026-05,5
5,2026-06,6
6,2026-07,56
7,2026-08,138
8,2026-09,205
9,2026-10,1056


,period,count
0,2026-01,26
1,2026-02,32
2,2026-03,6
3,2026-04,6
4,2026-05,2
5,2026-06,1
6,2026-07,14
7,2026-08,23
8,2026-09,20
9,2026-10,197


## 6. Replies Versus Top-Level Comments

In [6]:
reply_counts = reply_summary(comments)
reply_counts

,comment_type,count
0,top_level,1633
1,reply,367


## 7. Script Detection Summary

In [7]:
script_summary = pd.DataFrame({
    'feature': ['has_sinhala', 'has_latin', 'mixed_sinhala_latin', 'emoji_only'],
    'count': [
        int(comments_features['has_sinhala'].sum()),
        int(comments_features['has_latin'].sum()),
        int(comments_features['mixed_sinhala_latin'].sum()),
        int(comments_features['emoji_only'].sum()),
    ],
})
script_summary['percent'] = (script_summary['count'] / len(comments_features) * 100).round(2)
display(script_summary)
save_bar_chart(script_summary.set_index('feature')['count'], 'Script and Symbol Features', 'Comments', CHART_DIR / 'eda_script_features.png')

,feature,count,percent
0,has_sinhala,779,38.95
1,has_latin,915,45.75
2,mixed_sinhala_latin,147,7.35
3,emoji_only,439,21.95


PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/eda_script_features.png')

## 8. Potential Low-Information or Spam-Like Patterns

In [8]:
spam_summary = potential_spam_summary(comments_features)
repeated_comments = comments['comment_text'].fillna('').astype(str).value_counts().loc[lambda s: s > 1].head(10)
display(spam_summary)
display(repeated_comments.rename_axis('comment_text').reset_index(name='count'))

,metric,value
0,emoji_only,439
1,very_short_under_5_chars,300
2,repeated_comment_text_values,50
3,repeated_character_patterns,355


,comment_text,count
0,❤,52
1,❤❤,33
2,❤❤❤,27
3,❤❤❤❤,20
4,❤❤❤❤❤,19
5,❤❤❤❤❤❤,14
6,😂,13
7,❤️,10
8,😂😂😂,9
9,😂😂,6


## 9. Channel Imbalance and Language-Related Limitations
The pilot is intentionally balanced by selected channel count, but distribution can still differ by video coverage, reply share, date coverage, and text quality. Script detection does not identify language; it only marks Sinhala Unicode and Latin characters. A later labelled sample is needed to distinguish English, Singlish, Tanglish, Sinhala, spam, praise, requests, complaints, and true audience-demand comments.

In [9]:
eda_summary = pd.concat([
    channel_comments.assign(summary_table='comments_per_channel'),
], ignore_index=True)
script_summary.to_csv(ANALYSIS_DIR / 'eda_script_summary.csv', index=False)
spam_summary.to_csv(ANALYSIS_DIR / 'eda_spam_like_summary.csv', index=False)
per_video.to_csv(ANALYSIS_DIR / 'eda_comments_per_video.csv', index=False)
print('EDA outputs saved under', ANALYSIS_DIR.relative_to(PROJECT_ROOT))

EDA outputs saved under data/analysis/component_2
